In [1]:
%pip install -q google-colab-selenium[undetected]

In [11]:
import google_colab_selenium as gs
from selenium.webdriver.common.by import By
import pandas as pd
import re

In [7]:
driver = gs.UndetectedChrome()

<IPython.core.display.Javascript object>

In [4]:
def get_links_avtocod(driver, make):
    url = f'https://www.bibika.ru/search/mark_{make}?idtype=2'
    driver.get(url)

    links = []

    for a in driver.find_elements(By.TAG_NAME, 'a'):
      href = a.get_attribute('href')
      if href and '/catalog/' in href:
          links.append(href)

    return list(set(links))

In [5]:
get_links_avtocod(driver, 'bmw')

['http://www.bibika.ru/catalog/',
 'http://www.bibika.ru/catalog/_bmw/_x6/6007664',
 'http://www.bibika.ru/catalog/_bmw/_x5/6008328',
 'http://www.bibika.ru/catalog/_bmw/_3-series/6007658',
 'http://www.bibika.ru/catalog/_bmw/_520/6006437',
 'http://www.bibika.ru/catalog/_bmw/_x3/6007385',
 'http://www.bibika.ru/catalog/_bmw/_7-series/6007696',
 'http://www.bibika.ru/catalog/_bmw/_x5/6007076',
 'http://www.bibika.ru/catalog/_bmw/_3-series/6008144',
 'http://www.bibika.ru/catalog/_bmw/_428/6008365',
 'http://www.bibika.ru/catalog/_bmw/_x5/6007821',
 'http://www.bibika.ru/catalog/_bmw/_x6/6008282',
 'http://www.bibika.ru/catalog/_bmw/_3-series/6007838',
 'http://www.bibika.ru/catalog/_bmw/_x3/6004289',
 'http://www.bibika.ru/catalog/_bmw/_2-series/6006381',
 'http://www.bibika.ru/catalog/_bmw/_other/6008257',
 'http://www.bibika.ru/catalog/_bmw/_x3/6008275',
 'http://www.bibika.ru/catalog/_bmw/_3-series/6008364',
 'http://www.bibika.ru/catalog/_bmw/_7-series/6006954',
 'http://www.bibika

In [8]:
driver.get('https://www.bibika.ru/catalog/_ford/_focus/5994696')
html = driver.page_source

In [9]:
html

'<html><head><meta http-equiv="origin-trial" content="A7vZI3v+Gz7JfuRolKNM4Aff6zaGuT7X0mf3wtoZTnKv6497cVMnhy03KDqX7kBz/q/iidW7srW31oQbBt4VhgoAAACUeyJvcmlnaW4iOiJodHRwczovL3d3dy5nb29nbGUuY29tOjQ0MyIsImZlYXR1cmUiOiJEaXNhYmxlVGhpcmRQYXJ0eVN0b3JhZ2VQYXJ0aXRpb25pbmczIiwiZXhwaXJ5IjoxNzU3OTgwODAwLCJpc1N1YmRvbWFpbiI6dHJ1ZSwiaXNUaGlyZFBhcnR5Ijp0cnVlfQ=="><script type="text/javascript" async="" src="https://mc.yandex.ru/metrika/watch.js"></script><script src="//an.yandex.ru/system/context.js" type="text/javascript" async=""></script><script type="text/javascript" async="" charset="utf-8" src="https://www.gstatic.com/recaptcha/releases/kUYUkUlSyqkjTSMaN2w3RaOh/recaptcha__en.js" crossorigin="anonymous" integrity="sha384-p0z9Du3Dwf3ZTlFDRLUo9qWChfInAzC61KfZjxqsWFq5OGFzbrOOAw4oIQc6AMn7"></script><script type="text/javascript" src="//an.yandex.ru/system/context.js" async=""></script><script type="text/javascript" src="//an.yandex.ru/system/context.js" async=""></script><script type="text/javascript" 

In [12]:
def GetCar(url0, driver):

    driver.get(url0)

    car = {}
    car['url'] = url0
    try:
        title = driver.find_element(By.CLASS_NAME, 'card_top').find_element(By.TAG_NAME, 'h1')
        car['title'] = title.text.strip()
    except AttributeError:
        car['title'] = None

    if car['title']:
        year = re.findall(r'\d{4}', car['title'])
        if year:
            car['year'] = int(year[0])

    try:
        city = driver.find_element(By.CSS_SELECTOR, '.card_small_row.card_gor .small_row_right')
        car['city'] = city.text.strip()
    except AttributeError:
        car['city'] = None

    parts = url0.split('/')
    car['make'] = parts[-3][1:]
    car['model'] = parts[-2][1:]

    car['engine'] = None


    try:
        for elem in driver.find_elements(By.CSS_SELECTOR, '.card_small_row.bg_green'):
          if 'Двигатель' in elem.text:
              engine = elem.find_element(By.CLASS_NAME, 'small_row_right').text.split(',')
              car['engine'] = (engine[-1] + ', ' + engine[0]).strip()
              car['hp'] = int(engine[1].strip().split()[0])
    except AttributeError:
        car['engine'] = None

    car['transmission'] = None

    try:
        for elem in driver.find_elements(By.CSS_SELECTOR, '.card_tech_w'):
          if 'Привод' in elem.text:
              drive = elem.find_element(By.CLASS_NAME, 'card_prop').text.strip()
              car['drive'] = drive
          if 'Цена' in elem.text:
              price = elem.find_element(By.CLASS_NAME, 'card_prop').text.strip().split()
              car['price'] = int(price[0]+price[1])
    except AttributeError:
        pass

    try:
        for elem in driver.find_elements(By.CSS_SELECTOR, '.card_tech_g'):
          if 'Пробег' in elem.text:
              mileage = elem.find_element(By.CLASS_NAME, 'card_prop').text.strip().split()
              car['mileage'] = int(mileage[0])*1000
          if 'Кузов' in elem.text:
              body = elem.find_element(By.CLASS_NAME, 'card_prop').text.strip().split(',')
              car['body'] = body[0].lower()
              car['color'] = body[1].lower().strip()
    except AttributeError:
        pass

    car['wheel'] = None

    try:
        description = driver.find_element(By.CSS_SELECTOR, '.seller_comment').find_elements(By.CSS_SELECTOR, '.card_addblock p')
        car['description'] = ' '.join([paragraph.text.strip() for paragraph in description])
    except:
        car['description'] = None

    answer = {}

    answer['url'] = car.get('url')
    answer['title'] = car.get('title')
    answer['year'] = car.get('year')
    answer['city'] = car.get('city')
    answer['make'] = car.get('make')
    answer['model'] = car.get('model')
    answer['price'] = car.get('price')
    answer['engine'] = car.get('engine')
    answer['transmission'] = car.get('transmission')
    answer['mileage'] = car.get('mileage')
    answer['drive'] = car.get('drive')
    answer['body'] = car.get('body')
    answer['color'] = car.get('color')
    answer['wheel'] = car.get('wheel')
    answer['hp'] = car.get('hp')
    answer['description'] = car.get('description')

    return answer

GetCar('https://www.bibika.ru/catalog/_ford/_focus/5994696', driver)

{'url': 'https://www.bibika.ru/catalog/_ford/_focus/5994696',
 'title': 'Ford Focus 1.8d MT 116 л.с. 2002г за 330 тыс руб в Санкт-Петербурге',
 'year': 2002,
 'city': 'Санкт-Петербург',
 'make': 'ford',
 'model': 'focus',
 'price': 329900,
 'engine': 'дизель, 1.75 л.',
 'transmission': None,
 'mileage': 286000,
 'drive': 'передний',
 'body': 'универсал',
 'color': 'серебряный',
 'wheel': None,
 'hp': 116,
 'description': 'Цена автомобиля за наличные, без дополнительных платежей. Автомобиль в хорошем состоянии. Кузов без проблем, ржавчины и гнили нет, пороги живые, низа дверей тоже. Стаканы целые, без ремонтов. По технической части нареканий нет. Двигатель работает тихо, не дымит, не троит. Передачи все включаются без усилий, ничего не вылетает. Салон в отличном состоянии. По подвеске ничего не стучит, не бренчит. Электрика вся работает. Машина в целом полностью обслужена, все что нужно было заменить - менялось во время. Хорошая комплектация, есть все необходимое для ежедневной эксплуат

In [13]:
urls = []
urls.extend(get_links_avtocod(driver, 'bmw'))
urls.extend(get_links_avtocod(driver, 'ford'))
urls.extend(get_links_avtocod(driver, 'kia'))

In [14]:
len(urls)

94

In [15]:
from tqdm import tqdm

In [16]:
from selenium.common.exceptions import TimeoutException

selenium_cars = []

for i, link in enumerate(tqdm(urls)):
    driver = gs.UndetectedChrome()
    driver.set_page_load_timeout(20)

    try:
        car = GetCar(link, driver)
        selenium_cars.append(car)
    except TimeoutException:
        print(f'Слишком долго грузится, скипаем ссылку')
    except:
        print(link)

    driver.quit()

df_selenium_cars = pd.DataFrame(selenium_cars)
df_selenium_cars.to_csv(f'selenium_cars_final.csv', index=False)

  0%|          | 0/94 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

  1%|          | 1/94 [00:18<28:51, 18.62s/it]

http://www.bibika.ru/catalog/


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

 99%|█████████▉| 93/94 [14:50<00:06,  6.98s/it]

http://www.bibika.ru/catalog/_kia/_ceed/6006257


<IPython.core.display.Javascript object>

100%|██████████| 94/94 [14:59<00:00,  9.57s/it]
